In [ ]:
### import dependencies
import mne, os, pickle, glob
import numpy as np
import pandas as pd
from os.path import join
import matplotlib
%gui qt
mne.set_log_level(verbose='WARNING')
matplotlib.use('Qt5Agg')

In [ ]:
#========= Parameters =========#
# STC parameters
SNR = 3 # Usually the rule of thumb has been to set to 3 for ANOVAs, 2 for single trial analyses. 
method = "dSPM"
fixed = False # set to True if you want signed data. this will make this command run: mne.convert_forward_solution(fwd, surf_ori=True)
n_jobs = 4
lambda2 = 1.0 / SNR ** 2.0

#===========================#
expt = 'EXPT' # experiment name as written on the -raw.fif
ROOT = f'/path/to/{expt}' # change path to where your data is stored
os.chdir(ROOT)
# recommended folder structure.
epochs_dir = join(ROOT, 'epochs/')
evokeds_dir = join(ROOT, 'evokeds/')
subjects_dir = join(ROOT,'mri/') # MRI directory, only needed for source level stuff
raw_dir = join(ROOT, 'raw/')
meg_dir = join(ROOT, 'meg/')
log_dir  = join(ROOT, 'logs')
stc_dir = join(ROOT, 'stc/')

excluded = ['R0000']
## Use this when processing all the subjects
subjects = [i[:5] for i in os.listdir(raw_dir) if i.startswith('R') and i[:5] not in excluded and not i.endswith(".fif")] # List of subjects, can also be added manually
## Use this when processing some of the subjects
print('N =', len(subjects))
print(subjects)

In [ ]:
def get_epochs(subj, cond): # can also take mode as cond
    file_str = '%s_%s_%s-ica-epo.fif' % (subj, expt, cond)
    filename = os.path.join(epochs_dir, subj, file_str)
    epochs = mne.read_epochs(filename)
    
    return epochs

def get_evokeds(subj, epochs):
    evokeds = {}
    conditions = list(epochs.metadata.condition.unique()) #get conditions from epochs metadata
    for condition in conditions:
        ep = epochs[epochs.metadata["condition"]==condition] # get condition trials from epochs
        ev = ep.average()
        evokeds[condition] = ev #add ev to evokeds
    
    return evokeds

def get_source_space(subj, src_fname, force_new=False):
    print ('generating source space...')
    if (not os.path.isfile(src_fname)) or force_new:
        print ('src for subj = %s does not exist, creating file...' % (subj))
        src = mne.setup_source_space(subject=subj, spacing='ico4', subjects_dir=subjects_dir)
        src.save(src_fname, overwrite=True)
        #src = mne.read_source_spaces(src_fname)
        print ('done. file saved.')
    else:
        print('src for subj = %s already exists, loading file...' %subj)
        src = mne.read_source_spaces(src_fname)
        print('done.')

    return src      
    
def get_BEM(subj, bem_fname, force_new=False):
    bem = None
    print('getting bem')
    if (not os.path.isfile(bem_fname)) or force_new:
        print ('BEM for subj = %s does not exist, creating...' % (subj))
        conductivity = (0.3,) # for single layer
        model = mne.make_bem_model(subject=subj, ico=4, conductivity=conductivity, subjects_dir=subjects_dir)
        bem = mne.make_bem_solution(model)
        mne.write_bem_solution(bem_fname, bem, overwrite=True)
        print ('done. file saved.')
    
    return bem

def get_forward_solution(subj, info, src, trans, bem_fname, force_new=False):
    print('getting forward solution')
    fwd_path = os.path.join(epochs_dir, subj)
    fwd_fname = os.path.join(fwd_path, '%s-fwd.fif' %subj)
    if not os.path.exists(fwd_path):
        os.makedirs(fwd_path)
    if (not os.path.isfile(fwd_fname)) or force_new:
        print ('forward solution for subj = %s does not exist, creating file.' % (subj))
        fwd = mne.make_forward_solution(info=info, trans=trans, src=src, bem=bem_fname, ignore_ref=True)
        mne.write_forward_solution(fwd_fname, fwd, overwrite=True)
        fwd = mne.read_forward_solution(fwd_fname)
        print ('done.')
    else:
        print('fwd for subj = %s already exists, loading file...' %subj)
        fwd = mne.read_forward_solution(fwd_fname)
        print('done.')
        
    return fwd

def get_covariance_matrix(subj, epochs, force_new=False):
    conditions = list(epochs.metadata.condition.unique())
    mode = conditions[0][0]
    cov_fname = os.path.join(epochs_dir, subj, '%s_%s-cov.fif' %(subj, mode))
    print ('Getting covariance')
    if (not os.path.isfile(cov_fname)) or force_new:
        print('cov for subj = %s does not exist, creating file.' % (subj))
        cov = None
        cov = mne.compute_covariance(epochs, tmin=-0.1, tmax=0, 
                                         method=['shrunk', 'diagonal_fixed', 'empirical'])
        cov.save(cov_fname, overwrite=True)
        print ('done. file saved.')
    else:
        print('cov for subj = %s exists, loading file...' % (subj))
        cov = mne.read_cov(cov_fname)
        print('done.')

    return cov

def get_inverse_operator(info, fwd, cov):
    print ('getting inverse operator')
    if fixed == True:
        fwd = mne.convert_forward_solution(fwd, surf_ori=True)
    #fixed=False: Ignoring dipole direction.
    inv = mne.minimum_norm.make_inverse_operator(info, fwd, cov, depth=0.8, loose='auto', fixed=fixed) 
    lambda2 = 1.0 / SNR ** 2.0

    return inv, lambda2

def save_STC(stc_fsaverage, condition):
    stc_path = os.path.join(stc_dir, condition)
    if not os.path.exists(stc_path):
        os.mkdir(stc_path)
    stc_filename = os.path.join(stc_path, '%s_%s_dSPM' % (subj, condition))
    print(os.path.join(stc_dir, stc_filename))
    stc_fsaverage.save(os.path.join(stc_dir, stc_filename), overwrite=True)

def create_and_save_STCs(subj, evokeds, inv, lambda2):
    print ('%s: creating STCs...' % subj)
    for ev in evokeds:
        condition = ev
        evoked = evokeds[condition]
        stc = mne.minimum_norm.apply_inverse(evoked, inv, lambda2=lambda2, method='dSPM')
        morph = mne.compute_source_morph(stc, subject_from=subj, subject_to='fsaverage', 
                                        subjects_dir=subjects_dir, spacing=4)
        stc_fsaverage = morph.apply(stc)    
        save_STC(stc_fsaverage, condition)
        del stc, stc_fsaverage
    print ('DONE CREATING STCS FOR SUBJ = %s' %subj)


In [ ]:
for i, subj in enumerate(subjects):

    print("Computing STCs for subject (%s/%s)" % (str(i + 1), len(subjects)))

    trans_fname = os.path.join(meg_dir, subj, '%s-trans.fif' % subj)
    trans = mne.read_trans(trans_fname)
    src_fname = os.path.join(subjects_dir, subj, 'bem', '%s-ico-4-src.fif' % subj)
    src = get_source_space(subj, src_fname, force_new=False)
    bem_fname = os.path.join(subjects_dir, subj, 'bem', '%s-inner_skull-bem-sol.fif' % subj)

    epoch_file = os.path.join(meg_dir, subj, f'{subj}_{expt}-baselinecorr-ica-epo.fif')
    epochs = mne.read_epochs(epoch_file)
    
    info = epochs.info
    evokeds = get_evokeds(subj, epochs) # this should plot and save the ev to check
    bem = get_BEM(subj, bem_fname)
    fwd = get_forward_solution(subj, info, src, trans, bem_fname, force_new=False)
    cov = get_covariance_matrix(subj, epochs, force_new=False)
    inv, lambda2 = get_inverse_operator(info, fwd, cov)

    create_and_save_STCs(subj, evokeds, inv, lambda2)

    # delete variables
    del info, trans, src, fwd, cov, inv, evokeds